In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

def auditar_etiquetas_csv(ruta_bronce):
    """
    Recorre la Capa Bronce buscando archivos .csv de anotaciones.
    Extrae la duración y cantidad de cada condición (Etiqueta),
    evitando duplicados por canal.
    """
    directorio_base = Path(ruta_bronce)
    # Buscamos recursivamente todos los .csv
    archivos_csv = list(directorio_base.rglob("*.csv"))
    
    print(f"🔍 Escaneando {len(archivos_csv)} archivos CSV de anotaciones...")
    
    lista_df_eventos = []
    
    for archivo in archivos_csv:
        try:
            # Leer el CSV ignorando comentarios
            df = pd.read_csv(archivo, comment='#', header=None)
            
            # Asignar nombres de columnas según tu formato
            if len(df.columns) == 5:
                df.columns = ['Canal', 'Inicio_seg', 'Fin_seg', 'Etiqueta', 'Confianza']
            else:
                continue # Saltar si el formato no coincide
                
            # Forzar valores numéricos y limpiar nulos
            df[['Inicio_seg', 'Fin_seg']] = df[['Inicio_seg', 'Fin_seg']].apply(pd.to_numeric, errors='coerce')
            df = df.dropna(subset=['Inicio_seg', 'Fin_seg'])
            
            # MAGIA: Eliminar duplicados de canales para tener el evento temporal único
            eventos_unicos = df.drop_duplicates(subset=['Inicio_seg', 'Fin_seg', 'Etiqueta']).copy()
            
            # Calcular duración
            eventos_unicos['Duracion_seg'] = eventos_unicos['Fin_seg'] - eventos_unicos['Inicio_seg']
            eventos_unicos['Archivo'] = archivo.name
            eventos_unicos['Paciente'] = archivo.parents[3].name # Extrae 'aaaaaajy' de la ruta
            
            lista_df_eventos.append(eventos_unicos[['Paciente', 'Archivo', 'Etiqueta', 'Inicio_seg', 'Fin_seg', 'Duracion_seg']])
            
        except Exception as e:
            print(f"⚠️ Error procesando {archivo.name}: {e}")

    if not lista_df_eventos:
        print("❌ No se encontraron eventos válidos.")
        return None, None

    # Unir todo en un gran DataFrame maestro
    df_maestro = pd.concat(lista_df_eventos, ignore_index=True)
    
    # --- AGRUPACIÓN Y ESTADÍSTICAS POR CONDICIÓN ---
    resumen = df_maestro.groupby('Etiqueta').agg(
        Cantidad_Eventos=('Duracion_seg', 'count'),
        Tiempo_Total_Horas=('Duracion_seg', lambda x: x.sum() / 3600),
        Min_Segundos=('Duracion_seg', 'min'),
        Promedio_Segundos=('Duracion_seg', 'mean'),
        Mediana_Segundos=('Duracion_seg', 'median'),
        Max_Segundos=('Duracion_seg', 'max')
    ).reset_index()
    
    # Calcular el porcentaje de representación de cada clase respecto al tiempo total
    tiempo_total_absoluto = resumen['Tiempo_Total_Horas'].sum()
    resumen['Porcentaje_del_Dataset'] = (resumen['Tiempo_Total_Horas'] / tiempo_total_absoluto) * 100
    
    # Ordenar por Tiempo Total descendente
    resumen = resumen.sort_values(by='Tiempo_Total_Horas', ascending=False)
    
    # --- IMPRESIÓN DEL REPORTE ---
    print("\n" + "="*80)
    print("📈 RADIOGRAFÍA DE EVENTOS (DATASET TUSZ - CAPA BRONCE)")
    print("="*80)
    # Imprimir la tabla formateada
    print(resumen.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("="*80)
    print(f"Tiempo Total Analizado: {tiempo_total_absoluto:.2f} horas")
    
    return df_maestro, resumen

In [2]:
ruta_raw = r"D:\TUSZ_DataLake\01_Raw_Consolidated"
df_detallado, df_resumen = auditar_etiquetas_csv(ruta_raw)

🔍 Escaneando 7364 archivos CSV de anotaciones...

📈 RADIOGRAFÍA DE EVENTOS (DATASET TUSZ - CAPA BRONCE)
Etiqueta  Cantidad_Eventos  Tiempo_Total_Horas  Min_Segundos  Promedio_Segundos  Mediana_Segundos  Max_Segundos  Porcentaje_del_Dataset
    bckg             15452           1205.0973        0.1000           280.7630          150.0220     3598.0000                 88.0455
    fnsz              4904             80.5554        1.9844            59.1353           35.7596     3601.0000                  5.8855
    gnsz              2495             46.1076        2.5720            66.5280           43.2657     2448.0000                  3.3687
    cpsz              1520             30.4177        2.3355            72.0420           63.1489     1303.0000                  2.2223
    tcsz               199              4.4147       12.8930            79.8637           50.5827     2344.0492                  0.3225
    tnsz               126              0.6766        5.7697            19.3312 

In [4]:
df_detallado

,Paciente,Archivo,Etiqueta,Inicio_seg,Fin_seg,Duracion_seg
0,TUSZ_DataLake,aaaaatth_s001_t000.csv,bckg,0.0000,1289.0000,1289.0000
1,TUSZ_DataLake,aaaaatvk_s001_t000.csv,bckg,0.0000,1290.0000,1290.0000
2,TUSZ_DataLake,aaaaatvk_s002_t001.csv,bckg,0.0000,301.0000,301.0000
3,TUSZ_DataLake,aaaaatvk_s002_t002.csv,bckg,0.0000,301.0000,301.0000
4,TUSZ_DataLake,aaaaatvk_s002_t004.csv,bckg,0.0000,301.0000,301.0000
...,...,...,...,...,...,...
24872,TUSZ_DataLake,aaaaatdt_s004_t007.csv,bckg,0.0091,1.0009,0.9918
24873,TUSZ_DataLake,aaaaatdt_s004_t008.csv,bckg,0.0182,1.0009,0.9827
24874,TUSZ_DataLake,aaaaatdt_s004_t010.csv,bckg,0.0182,1.0191,1.0009
24875,TUSZ_DataLake,aaaaatdt_s004_t011.csv,bckg,0.0000,1.0191,1.0191


In [5]:
df_resumen

,Etiqueta,Cantidad_Eventos,Tiempo_Total_Horas,Min_Segundos,Promedio_Segundos,Mediana_Segundos,Max_Segundos,Porcentaje_del_Dataset
1,bckg,15452,1205.097278,0.1000,280.763021,150.02205,3598.0000,88.045541
3,fnsz,4904,80.555361,1.9844,59.135257,35.75955,3601.0000,5.885450
4,gnsz,2495,46.107631,2.5720,66.528044,43.26570,2448.0000,3.368667
2,cpsz,1520,30.417728,2.3355,72.041987,63.14895,1303.0000,2.222348
7,tcsz,199,4.414686,12.8930,79.863673,50.58270,2344.0492,0.322541
8,tnsz,126,0.676591,5.7697,19.331160,17.09395,146.8889,0.049432
6,spsz,57,0.637037,20.1400,40.233942,33.81600,97.0608,0.046543
5,mysz,3,0.549933,606.0000,659.919333,686.86900,686.8890,0.040179
0,absz,121,0.264154,2.3640,7.859126,6.49720,26.4343,0.019299


In [9]:
import pandas as pd

MAPEO_MACRO_CLASES = {
    'bckg': 0, 
    
    # Macro-Clase 1: Focales (8.15% del dataset)
    'fnsz': 1, 'cpsz': 1, 'spsz': 1, 
    
    # Macro-Clase 2: Generalizadas (3.80% del dataset)
    'gnsz': 2, 'tcsz': 2, 'tnsz': 2, 'absz': 2, 'mysz': 2
}

NOMBRES_CLASES = {
    0: '0_Fondo',
    1: '1_Focal',
    2: '2_Generalizada'
}

# 2. Aplicar el mapeo a df_detallado
# Si encuentra una etiqueta rara que no está en el diccionario, le asigna 3 (Otras) por defecto
df_detallado['Macro_Clase_ID'] = df_detallado['Etiqueta'].map(MAPEO_MACRO_CLASES).fillna(3).astype(int)
df_detallado['Macro_Clase_Nombre'] = df_detallado['Macro_Clase_ID'].map(NOMBRES_CLASES)

# 3. Generar el nuevo resumen agrupado por las 4 Macro-Clases
resumen_macro = df_detallado.groupby(['Macro_Clase_ID', 'Macro_Clase_Nombre']).agg(
    Cantidad_Eventos=('Duracion_seg', 'count'),
    Tiempo_Total_Horas=('Duracion_seg', lambda x: x.sum() / 3600),
    Min_Segundos=('Duracion_seg', 'min'),
    Promedio_Segundos=('Duracion_seg', 'mean'),
    Max_Segundos=('Duracion_seg', 'max')
).reset_index()

# Calcular el porcentaje del dataset
tiempo_total = resumen_macro['Tiempo_Total_Horas'].sum()
resumen_macro['Porcentaje_Dataset'] = (resumen_macro['Tiempo_Total_Horas'] / tiempo_total) * 100

print("\n" + "="*80)
print("📊 DISTRIBUCIÓN DE MACRO-CLASES")
print("="*80)
print(resumen_macro.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


📊 DISTRIBUCIÓN DE MACRO-CLASES
 Macro_Clase_ID Macro_Clase_Nombre  Cantidad_Eventos  Tiempo_Total_Horas  Min_Segundos  Promedio_Segundos  Max_Segundos  Porcentaje_Dataset
              0            0_Fondo             15452           1205.0973        0.1000           280.7630     3598.0000             88.0455
              1            1_Focal              6481            111.6101        1.9844            61.9961     3601.0000              8.1543
              2     2_Generalizada              2944             52.0130        2.3640            63.6028     2448.0000              3.8001
